In [1]:
import pandas as pd
import sqlite3

# Chargement
conso = pd.read_csv("./data/brest_metropole_daily.csv")
meteo = pd.read_csv("./data/brest_meteo_daily.csv")
pop = pd.read_csv("./data/brest_population_daily_est.csv")
cal = pd.read_csv("./data/brest_calendar_off_days.csv")

# Uniformisation des dates
for df in [conso, meteo, pop, cal]:
    df["date"] = pd.to_datetime(df["date"])

# Jointure
timeserie = (
    conso[["date", "conso_mwh"]]
    .merge(
        meteo[["date", "temp_moy_c"]],
        on="date",
        how="inner"
    )
    .merge(
        pop[["date", "population_daily_est"]],
        on="date",
        how="inner"
    )
    .merge(
        cal[["date", "is_off"]],
        on="date",
        how="inner"
    )
)

# Renommage
timeserie = timeserie.rename(columns={
    "conso_mwh": "consommation_mwh",
    "population_daily_est": "population",
    "is_off": "week_end_ferie"
})

# Booléen SQLite
timeserie["week_end_ferie"] = timeserie["week_end_ferie"].astype(int)

# Format date ISO
timeserie["date"] = timeserie["date"].dt.strftime("%Y-%m-%d")

# Export SQLite
conn = sqlite3.connect("timeserie.db")

timeserie.to_sql(
    "timeserie",
    conn,
    if_exists="replace",
    index=False
)

conn.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS idx_timeserie_date
ON timeserie(date)
""")

conn.commit()
conn.close()

In [5]:
import sqlite3
import pandas as pd

# Saisie utilisateur
temperature = float(input("Température moyenne (°C) : "))
is_off = input("Week-end/Férié ? (true/false) : ").strip().lower() == "true"

conn = sqlite3.connect("timeserie.db")

query = """
SELECT
    date,
    consommation_mwh,
    temp_moy_c,
    week_end_ferie
FROM timeserie
WHERE temp_moy_c BETWEEN ? AND ?
  AND week_end_ferie = ?
ORDER BY date
"""

resultats = pd.read_sql_query(
    query,
    conn,
    params=(
        temperature - 0.2,
        temperature + 0.2,
        int(is_off)
    )
)

conn.close()

print(f"{len(resultats)} jours trouvés")
display(resultats)

# Liste seule des consommations
consommations = resultats["consommation_mwh"].tolist()

print("\nConsommations :")
print(consommations)

print("Min :", resultats["consommation_mwh"].min())
print("Moyenne :", resultats["consommation_mwh"].mean())
print("Médiane :", resultats["consommation_mwh"].median())
print("Max :", resultats["consommation_mwh"].max())
print("Écart-type :", resultats["consommation_mwh"].std())

65 jours trouvés


,date,consommation_mwh,temp_moy_c,week_end_ferie
0,2020-04-09,2215.50,14.1,0
1,2020-04-16,2125.00,14.2,0
2,2020-05-05,2257.25,13.9,0
3,2020-06-05,2227.25,14.0,0
4,2020-06-08,2236.50,14.0,0
...,...,...,...,...
60,2025-10-21,2668.75,13.9,0
61,2025-11-03,3067.75,14.1,0
62,2025-11-06,2891.50,14.0,0
63,2026-04-29,2374.00,13.8,0



Consommations :
[2215.5, 2125.0, 2257.25, 2227.25, 2236.5, 2310.75, 2355.0, 2290.75, 2231.25, 2295.75, 2645.75, 2690.0, 2806.0, 2916.25, 2299.2, 2287.25, 2352.75, 2329.0, 2576.5, 2510.5, 1931.25, 1869.5, 1921.25, 1753.0, 1850.5, 1796.75, 1863.75, 1751.75, 2034.0, 2558.75, 2375.0, 3072.5, 3148.0, 2416.0, 2232.0, 2080.25, 2210.25, 2372.75, 2224.0, 2624.5, 2285.5, 2308.0, 2246.0, 2185.0, 2184.5, 2254.75, 2263.75, 2329.5, 2355.75, 2483.0, 2699.75, 2727.5, 3162.25, 2363.5, 2203.75, 2246.75, 2276.25, 2301.0, 2445.25, 2633.5, 2668.75, 3067.75, 2891.5, 2374.0, 2597.0]
Min : 1751.75
Moyenne : 2363.049230769231
Médiane : 2301.0
Max : 3162.25
Écart-type : 322.71446133398877
